In [2]:
import numpy as np
import pandas as pd

# беру тимохин файл
race = pd.read_csv("../data/data_final/race_features.csv")



# таргет это финишная позиция как класс (1, 2,3 .. 20)
# берем position_filled, там сходы уже проставлены последним местом и нет пустот
race["target_position"] = race["position_filled"].astype(int)

print("загружено:", race.shape)
print()
print("сколько раз встречается каждая позиция:")
print(race["target_position"].value_counts().sort_index())

загружено: (1397, 44)

сколько раз встречается каждая позиция:
target_position
1      70
2      70
3      70
4      70
5      70
6      70
7      70
8      70
9      70
10     70
11     70
12     70
13     70
14     70
15     68
16     61
17     56
18     43
19     31
20    158
Name: count, dtype: int64


у тимофея модели предсказывают конкретную позицию (регрессия). а мы хотим по другому: чтобы модель выдавала вероятность каждого места. тогда из этих вероятностей можно собрать любой топ-N (шанс топ-3, топ-5 и тд) просто складывая. это удобно для бота, можно показать "вероятность подиума 60 процентов" вместо сухого "будет 3 место"

In [3]:
#признаки берем те же, чтобы все было по честному и сравнимо
#тут только то что известно ДО гонки, никакой утечки
num_features = ["quali_position", "driver_form_5", "driver_form_all", "team_form_5", "driver_track_history", "driver_finish_rate", "driver_experience", "driver_speed_5", "is_rookie", "field_size", "air_temp_mean", "track_temp_mean", "humidity_mean", "wind_speed_mean", "rainfall_max"]
cat_features = ["team_name", "circuit_short_name", "circuit_type"]
target = "target_position"

print("числовых:", len(num_features), "категориальных:", len(cat_features))

числовых: 15 категориальных: 3


In [4]:
# делим по времени: учимся на 2023-2024, проверяем на 2025
#случайно мешать нельзя, иначе модель будет подсматривать будущее
train = race[race["year"].isin([2023, 2024])].reset_index(drop=True)
test = race[race["year"] == 2025].reset_index(drop=True)

y_train = train[target]
y_test = test[target]

print("train:", train.shape, "гонок", train["session_key"].nunique())
print("test:", test.shape, "гонок", test["session_key"].nunique())

train: (918, 44) гонок 46
test: (479, 44) гонок 24


In [5]:
# категории (команда, трасса, тип трассы) превращаем в нули и единицы через get_dummies
X_all = pd.get_dummies(race[num_features + cat_features], columns=cat_features, drop_first=True)

X_train = X_all[race["year"].isin([2023, 2024])].reset_index(drop=True)
X_test = X_all[race["year"] == 2025].reset_index(drop=True)

print("признаков после кодирования:", X_all.shape[1])

признаков после кодирования: 49


In [6]:
from catboost import CatBoostClassifier

# мультиклассовый катбуст, на выходе вероятность каждой позиции для каждого пилота
model = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=4, random_seed=13, verbose=0)
model.fit(X_train, y_train)

# predict_proba как раз и дает вероятности по всем позициям
proba = model.predict_proba(X_test)
print("форма вероятностей:", proba.shape)
print("классы (позиции):", model.classes_)

форма вероятностей: (479, 20)
классы (позиции): [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]


In [7]:
# вот эта функция и дает гибкость
# берем вероятности всех позиций пилота и складываем те что попадают в топ-N
# например шанс топ-3 это вероятность 1 места плюс 2 плюс 3
def get_topn_probability(proba_row, classes, n):
    total = 0.0
    for i in range(len(classes)):
        if classes[i] <= n:
            total = total + proba_row[i]
    return total


# проверим на первом пилоте, считаем разные топы
first_proba = proba[0]
print("вероятность топ-1:", round(get_topn_probability(first_proba, model.classes_, 1), 3))
print("вероятность топ-3:", round(get_topn_probability(first_proba, model.classes_, 3), 3))
print("вероятность топ-5:", round(get_topn_probability(first_proba, model.classes_, 5), 3))
print("вероятность топ-10:", round(get_topn_probability(first_proba, model.classes_, 10), 3))

вероятность топ-1: 0.006
вероятность топ-3: 0.05
вероятность топ-5: 0.117
вероятность топ-10: 0.377


теперь соберем прогноз на целую гонку, по всем пилотам сразу, и отсортируем по шансу победы

In [8]:
# по конкретной гонке собираем табличку: пилот, его шансы и где он реально финишировал
def show_race_prediction(session_key, proba, test_df, classes):
    mask = test_df["session_key"] == session_key
    idx = test_df[mask].index

    rows = []
    for i in idx:
        driver = test_df.loc[i, "name_acronym"]
        team = test_df.loc[i, "team_name"]
        real_pos = test_df.loc[i, "position"]
        p1 = get_topn_probability(proba[i], classes, 1)
        p3 = get_topn_probability(proba[i], classes, 3)
        p10 = get_topn_probability(proba[i], classes, 10)
        rows.append([driver, team, real_pos, p1, p3, p10])

    result = pd.DataFrame(rows, columns=["пилот", "команда", "факт_место", "P(топ1)", "P(топ3)", "P(топ10)"])
    # сортируем по шансу победы, фаворит сверху
    result = result.sort_values("P(топ1)", ascending=False).reset_index(drop=True)
    return result


# берем первую гонку из теста и смотрим прогноз
first_session = test["session_key"].iloc[0]
prediction = show_race_prediction(first_session, proba, test, model.classes_)
print("прогноз на гонку, session_key:", first_session)
prediction.head(20)

прогноз на гонку, session_key: 9693


,пилот,команда,факт_место,P(топ1),P(топ3),P(топ10)
0,NOR,McLaren,1.0,0.300431,0.597038,0.911840
1,VER,Red Bull Racing,2.0,0.148522,0.476726,0.902339
2,PIA,McLaren,9.0,0.141321,0.464888,0.867463
3,RUS,Mercedes,3.0,0.057654,0.336883,0.833729
4,LEC,Ferrari,8.0,0.044711,0.225974,0.834912
5,HAM,Ferrari,10.0,0.043293,0.196725,0.790900
6,SAI,Williams,NaN,0.016594,0.088694,0.611115
7,ALO,Aston Martin,NaN,0.007530,0.046010,0.481810
8,TSU,Racing Bulls,12.0,0.006648,0.056616,0.631530
9,ANT,Mercedes,4.0,0.006118,0.050276,0.376946


посмотрим насколько модель хороша на всем тесте, а не на одной удачной гонке. считаем три метрики

In [9]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

# для каждого пилота считаем вероятности топ-1, топ-3, топ-10
test_eval = test.copy().reset_index(drop=True)

p_top1 = []
p_top3 = []
p_top10 = []
for i in range(len(test_eval)):
    p_top1.append(get_topn_probability(proba[i], model.classes_, 1))
    p_top3.append(get_topn_probability(proba[i], model.classes_, 3))
    p_top10.append(get_topn_probability(proba[i], model.classes_, 10))

test_eval["p_top1"] = p_top1
test_eval["p_top3"] = p_top3
test_eval["p_top10"] = p_top10

# реальные метки, попал пилот в топ или нет
test_eval["real_top1"] = (test_eval["position"] == 1).astype(int)
test_eval["real_top3"] = (test_eval["position"] <= 3).astype(int)
test_eval["real_top10"] = (test_eval["position"] <= 10).astype(int)

# roc-auc,average,losslog 
metrics_rows = []
for name, real, p in [("топ-1", "real_top1", "p_top1"), ("топ-3", "real_top3", "p_top3"), ("топ-10", "real_top10", "p_top10")]:
    auc = roc_auc_score(test_eval[real], test_eval[p])
    ap = average_precision_score(test_eval[real], test_eval[p])
    ll = log_loss(test_eval[real], test_eval[p])
    metrics_rows.append([name, round(auc, 3), round(ap, 3), round(ll, 3)])

metrics_table = pd.DataFrame(metrics_rows, columns=["задача", "ROC-AUC", "Avg Precision", "Log-loss"])
metrics_table

,задача,ROC-AUC,Avg Precision,Log-loss
0,топ-1,0.977,0.730,0.098
1,топ-3,0.955,0.824,0.205
2,топ-10,0.844,0.855,0.490


а это уже почти бот: вводим гонку и пилота, получаем табличку вероятностей по каждой позиции

In [10]:
# вводим session_key гонки и код пилота (например NOR), получаем его шансы на каждое место
def predict_driver(session_key, driver_acronym, proba, test_df, classes):
    mask = (test_df["session_key"] == session_key) & (test_df["name_acronym"] == driver_acronym)
    rows = test_df[mask]

    if len(rows) == 0:
        print("пилот не найден на этой гонке")
        return None

    i = rows.index[0]

    driver = test_df.loc[i, "name_acronym"]
    team = test_df.loc[i, "team_name"]
    real_pos = test_df.loc[i, "position"]
    print("пилот:", driver, "(" + str(team) + ")")
    print("реальное место:", real_pos)
    print()

    # табличка вероятность на каждую позицию
    result = pd.DataFrame({"позиция": classes, "вероятность_%": (proba[i] * 100).round(2)})
    return result


first_session = test["session_key"].iloc[0]
predict_driver(first_session, "NOR", proba, test, model.classes_)

пилот: NOR (McLaren)
реальное место: 1.0



,позиция,вероятность_%
0,1,30.04
1,2,17.96
2,3,11.70
3,4,7.64
4,5,5.41
5,6,9.28
6,7,2.31
7,8,2.10
8,9,1.15
9,10,3.60




теперь попробуем нейросеть и сравним ее с катбустом. сразу скажем: данных у нас мало (около 900 строк на обучение), а нейросетям нужно сильно больше, так что скорее всего катбуст победит



In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

# нейросети обязательно нужен маштаб, иначе обучение разваливается (деревьям было пофиг)
#учим scaler только на трейне, чтобы не подсматривать тест
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# torch хочет классы с нуля, а у нас позиции с 1. поэтому отнимаем 1 (потом прибавим обратно)
y_train_nn = (y_train - 1).values
y_test_nn = (y_test - 1).values

# переводим numpy в тензоры торча
X_train_t = torch.tensor(X_train_sc, dtype=torch.float32)
X_test_t = torch.tensor(X_test_sc, dtype=torch.float32)
y_train_t = torch.tensor(y_train_nn, dtype=torch.long)
y_test_t = torch.tensor(y_test_nn, dtype=torch.long)

print("X_train_t:", X_train_t.shape)
print("число классов:", y_train_t.max().item() + 1)

X_train_t: torch.Size([918, 49])
число классов: 20


In [ ]:

class RaceNet(nn.Module):
    def __init__(self, n_features, n_classes):
        super().__init__()
     
        self.fc1 = nn.Linear(n_features, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.3)
      
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.3)
    
        self.fc3 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.act1(x)
        x = self.drop1(x)
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.act2(x)
        x = self.drop2(x)
        x = self.fc3(x)
        return x


model_nn = RaceNet(n_features=49, n_classes=20)
print(model_nn)

RaceNet(
  (fc1): Linear(in_features=49, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act1): ReLU()
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (act2): ReLU()
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=20, bias=True)
)


In [ ]:
import torch.optim as optim

# функция потерь для классификации и оптимизатор adam (тоже что и для катбуста, чтобы сравнивать модели по честному)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

# крутим обучение 200 эпох, на каждой считаем ошибку и подкручиваем веса
n_epochs = 200
for epoch in range(n_epochs):
    model_nn.train()
    optimizer.zero_grad()

    outputs = model_nn(X_train_t)
    loss = criterion(outputs, y_train_t)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print("эпоха", epoch + 1, "из", n_epochs, "loss", round(loss.item(), 4))

эпоха 20 из 200 loss 2.7694
эпоха 40 из 200 loss 2.5307
эпоха 60 из 200 loss 2.4317
эпоха 80 из 200 loss 2.318
эпоха 100 из 200 loss 2.2339
эпоха 120 из 200 loss 2.1242
эпоха 140 из 200 loss 2.0055
эпоха 160 из 200 loss 1.9752
эпоха 180 из 200 loss 1.9086
эпоха 200 из 200 loss 1.8077


In [ ]:
import torch.nn.functional as F

model_nn.eval()

with torch.no_grad():
    logits = model_nn(X_test_t)
    proba_nn = F.softmax(logits, dim=1).numpy()

nn_classes = np.arange(1, 21)

print("форма вероятностей:", proba_nn.shape)
print("сумма вероятностей у первого пилота:", round(proba_nn[0].sum(), 3))

форма вероятностей: (479, 20)
сумма вероятностей у первого пилота: 1.0


собираем сравнение катбуста и нейросети в одну таблицу

In [15]:
# реальные метки топ-N для теста
real_top = {}
real_top[1] = (test["position"] == 1).astype(int)
real_top[3] = (test["position"] <= 3).astype(int)
real_top[10] = (test["position"] <= 10).astype(int)


# для любой модели и любого топа считаем auc и average precision
def model_scores(proba_matrix, classes, n):
    probs = []
    for i in range(len(proba_matrix)):
        probs.append(get_topn_probability(proba_matrix[i], classes, n))
    auc = roc_auc_score(real_top[n], probs)
    ap = average_precision_score(real_top[n], probs)
    return round(auc, 3), round(ap, 3)


# собираем таблицу, метрики обеих моделей рядом
rows = []
for n in [1, 3, 10]:
    cat_auc, cat_ap = model_scores(proba, model.classes_, n)
    nn_auc, nn_ap = model_scores(proba_nn, nn_classes, n)
    rows.append(["топ-" + str(n), cat_auc, cat_ap, nn_auc, nn_ap])

compare = pd.DataFrame(rows, columns=["задача", "CatBoost AUC", "CatBoost AP", "NeuralNet AUC", "NeuralNet AP"])
compare

,задача,CatBoost AUC,CatBoost AP,NeuralNet AUC,NeuralNet AP
0,топ-1,0.977,0.730,0.947,0.334
1,топ-3,0.955,0.824,0.926,0.691
2,топ-10,0.844,0.855,0.809,0.835


Вывод: катбуст обходит нейросеть почти везде, особенно на топ-1 по average precision (предсказать единственного победителя сложнее всего). это логично, нейросетям нужно много данных, а у нас всего около 900 строк. на маленьких табличных данных градиентный бустинг почти всегда сильнее нейросетей.

поэтому в боте используем катбуст как основную модель, а нейросеть оставляем для кайфа